In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import json
from mtrain.utils import globL, mkdir
from itertools import batched
import shutil
from pathlib import Path
from tqdm import tqdm
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import cv2
import random
from mtrain.utils import show, DiskImage, DiskBooleanMask, overlay_mask_on_img as OV
from mtrain.example_dir import ExampleDir, load_npz
from mtrain.example_dir.iterdir import get_dirs
from mtrain.example_dir.defaults import default_negmask_learners, default_smallnet_learners
from tqdm import tqdm

In [ ]:
DELHI_ALL_DATA = Path("/Users/hariomnarang/Desktop/personal/roads/mapillary_downloader/data/delhi/images")
CHUNKS_DEST = Path("/Users/hariomnarang/Desktop/personal/roads/datasets/inference/delhi/chunks")

# make chunks

In [ ]:

images = globL(DELHI_ALL_DATA, "*.jpg")
len(images)

In [ ]:
chunk_size = 500
batches = list(batched(images, chunk_size))
for i, batch in enumerate(tqdm(batches)):
    for image_path in batch:
        chunk_dest = mkdir(CHUNKS_DEST / str(i))
        dest = mkdir(chunk_dest / image_path.stem)
        shutil.copy(image_path, dest / "image.jpg")

In [ ]:
CHUNKS_DEST

In [ ]:
for i in range(66):
    ! (cd /Users/hariomnarang/Desktop/personal/roads/datasets/inference/delhi/chunks && dvctar add {i})

# run on chunks

In [ ]:
dirs = list(get_dirs(CHUNKS_DEST / "5"))

In [ ]:
negmask = default_negmask_learners(Path("/Users/hariomnarang/Desktop/personal/roads/datasets/models"), ["md", "high-recall", "unblurred"], 4)
smallnet = default_smallnet_learners(Path("/Users/hariomnarang/Desktop/personal/roads/datasets/models"), ["md", "sm"], 4)


In [ ]:
edirs = [ExampleDir(d, smallnet, negmask) for d in dirs]

In [ ]:
from tqdm import tqdm
for edir in tqdm(edirs):
    edir.smallnet_mask_path("md")
    edir.smallnet_mask_path("sm")
    edir.negmask_paths("md", "md")
    edir.negmask_paths("high-recall", "md")
    edir.negmask_paths("unblurred", "sm")

# speed using segformer masks

In [ ]:
negmask = default_negmask_learners(Path("/Users/hariomnarang/Desktop/personal/roads/datasets/models"), ["md", "high-recall", "unblurred"], 4)
smallnet = default_smallnet_learners(Path("/Users/hariomnarang/Desktop/personal/roads/datasets/models"), ["md", "sm"], 4)
dirs = globL("/Users/hariomnarang/Desktop/personal/roads/datasets/inference/delhi/chunks/0/", "*")

edirs = [ExampleDir(d, smallnet, negmask) for d in dirs]
# dirs

In [ ]:
smallnet["md"].bs = 8

In [ ]:
ExampleDir.batch_predict_smallnet_masks(smallnet["md"], edirs[:8], 4, True)

In [ ]:
img = edir.load_and_resize_image(edir.image_path)

In [ ]:
from tqdm import tqdm
from itertools import batched
from mtrain.example_dir.mapi_cons import MAPI_LABELS_TO_EXCLUDE, ELEV_LABELS_TO_EXCLUDE
from pathlib import Path
from mtrain.seg import mapillary as mapi, elevated_vegetation as elev

elev_pred = DiskBooleanMask.load(edir.elev_mask_path())
mapi_pred = DiskBooleanMask.load(edir.mapi_mask_path())

if mapi_pred is not None:
    mapi_exclude_mask = mapi.get_mask_with_labels(
        mapi_pred, MAPI_LABELS_TO_EXCLUDE
    )
else:
    mapi_exclude_mask = np.zeros(img.shape[:2], dtype=bool)

if elev_pred is not None:
    elev_exclude_mask = elev.get_mask_with_labels(
        elev_pred, ELEV_LABELS_TO_EXCLUDE
    )
else:
    elev_exclude_mask = np.zeros(img.shape[:2], dtype=bool)

In [ ]:
total_exclude_mask = mapi_exclude_mask | elev_exclude_mask
kernel = np.ones((5,5), np.uint8)
total_exclude_mask = cv2.dilate(total_exclude_mask.astype(np.uint8), kernel, iterations=5).astype(bool)

cut = img.copy()
cut[total_exclude_mask] = 0

show([cut, img])

In [ ]:
%%timeit

smallnet["md"].predict(cut)

In [ ]:
%%timeit

smallnet["md"].predict(img)
# edir.smallnet_mask_path("md", True)
# edir.trimmed_mask_path("md")